# Tutorial 1: Basic Python Integration

## Goals

- Introduce DataStructure, Filter, Parameter, DataPath
- Run basic python code to execute a filter
- Visualize a DataArray using MatPlotLib


## Import Statements

In [29]:
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import simplnx as nx
import itkimageprocessing as nxitk
import orientationanalysis as nxor

## Utility Function

We can use this function to print the any warnings and/or errors from the filter and to stop further execution on failure.

In [30]:
def check_filter_result(filter: nx.IFilter, result: nx.IFilter.ExecuteResult) -> None:
  if len(result.warnings) != 0:
    print(f'{filter.name()} ::  Warnings: {result.warnings}')
  
  has_errors = len(result.errors) != 0 
  if has_errors:
    print(f'{filter.name()} :: Errors: {result.errors}')
    raise RuntimeError(result)
  
  print(f"{filter.name()} :: No errors running the filter")


## Executing a filter

In [31]:
# Create the DataStructure Object
data_structure = nx.DataStructure()
# Create the Filter object
nx_filter = nxor.ReadAngDataFilter()
# Execute the filter
result = nx_filter.execute(
   data_structure=data_structure,
   cell_attribute_matrix_name="Cell Data",
   cell_ensemble_attribute_matrix_name="Cell Ensemble Data",
   output_image_geometry_path=nx.DataPath("DataContainer"),
   input_file="Data/Small_IN100/Slice_1.ang"
)
check_filter_result(nx_filter, result)

nx::core::ReadAngDataFilter :: No errors running the filter


## Accessing NumPy Arrays

Use the `npview()` function to get a **view** into the DataArray. This is **NOT** a copy 
of the data. 

- Views all access the same underlying chunk of memory
- Copies will allocate a new chunk of memory and copy the data into that

[NumPy copy vs view docs](https://numpy.org/doc/stable/user/basics.copies.html)

The data we are reading will have some erroneous negative values for the "Confidence Index" array.

We are going to fix this issue by constraining all values to be at least 0 using [numpy.clip](https://numpy.org/doc/stable/reference/generated/numpy.clip.html).

In [32]:
# Get a numpy view of the Confidence Index Array and ensure all dimensions are
# are > 1
ci_view: np.ndarray = data_structure["DataContainer/Cell Data/Confidence Index"].npview()

# Ensure all values are at least 0
np.clip(ci_view, a_min=0, a_max=None, out=ci_view)

array([[[[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]],

        [[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]],

        [[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]],

        ...,

        [[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]],

        [[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]],

        [[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]]]], shape=(1, 201, 189, 1), dtype=float32)

## Plotting

Let's use Matplotlib to render the EBSD Confidence Index array.

Before plotting we must use [`squeeze()`](https://numpy.org/doc/stable/reference/generated/numpy.squeeze.html).
DREAM3D will have some dimensions that are 1.

- For example, a single component array such as "Confidence Index"
- A single layer of an Image Geometry with Z dimension = 1

Matplotlib accepts array-like data with a shape of `(M, N)`. `squeeze()` removes axes of length one allowing us to plot our data.

In [ ]:
print(f'DREAM3D Dimensions: {ci_view.shape}')

ci_view = ci_view.squeeze()

print(f'NumPy Dimensions: {ci_view.shape}')

In [ ]:
# Show the result
image_plot = plt.imshow(ci_view)
colorbar = plt.colorbar(image_plot)
colorbar.set_label('Color Scale')
plt.title("Corrected Confidence Index")
plt.axis('on')  # to turn off axes
plt.show()

## Questions?